# 调试 MedQA 评估流程

这是一个用于调试 MedQA 评估流程的 Jupyter Notebook。它将原始脚本分解为模块化步骤，允许用户检查中间变量、验证 Prompt 格式、查看模型的原始生成文本，并测试答案提取正则表达式的准确性。

In [ ]:
import json
import re
import sys
import argparse
from typing import Optional, List, Tuple

import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# 设置随机种子以确保结果可复现
torch.manual_seed(42)
print("Environment and libraries loaded successfully.")

In [ ]:
# 配置参数
# 注意：请根据实际情况修改 MODEL_PATH 和 DATASET_PATH
MODEL_PATH = "/data/ocean/decoding/model/Qwen/Qwen3-14B"  # 替换为你的模型路径
DATASET_PATH = "./phrases_no_exclude_test.jsonl"  # 替换为你的数据集路径
OUTPUT_FILE = "debug_output.json"

BATCH_SIZE = 1  # 调试时建议设为 1
MAX_NEW_TOKENS = 512 # 稍微减小以便快速测试，原始是 4096
TEMPERATURE = 0.6
TOP_P = 0.9

print(f"Config:\n Model: {MODEL_PATH}\n Dataset: {DATASET_PATH}\n Temp: {TEMPERATURE}")

In [ ]:
def load_dataset(dataset_path: str):
    """加载 JSONL 格式的数据集"""
    data = []
    try:
        with open(dataset_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
        print(f"Successfully loaded {len(data)} items from {dataset_path}")
    except FileNotFoundError:
        print(f"Error: File not found at {dataset_path}. using dummy data for demo.")
        # Dummy data for demonstration if file missing
        data = [
            {
                "question": "A 25-year-old man presents...",
                "options": {"A": "Ataxia", "B": "Conjunctival injection", "C": "Miosis", "D": "Nystagmus"},
                "answer_idx": "C"
            }
        ]
    return data

dataset = load_dataset(DATASET_PATH)

# Check first item
if dataset:
    print("\n--- Sample Item 0 ---")
    print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

In [ ]:
def load_model(model_path: str):
    print(f"Loading model from {model_path}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            trust_remote_code=True,
            torch_dtype="auto",
            device_map="auto",
        )
        model.eval()
        print("Model loaded successfully.")
        return tokenizer, model
    except OSError:
        # Fallback for demo without weights
        print("Warning: Could not load actual model. Using placeholder for structure check.")
        return None, None

# Uncomment to actually load (takes time/memory)
# tokenizer, model = load_model(MODEL_PATH)
print("Skipping actual model load to save time in this notebook preview. Set 'tokenizer, model' properly to run.")
# Mock for subsequent cells to not crash
tokenizer, model = None, None

In [ ]:
def create_prompt(question, options) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])
    prompt = f"""You are a medical expert. Please analyze the following medical question step by step and provide your reasoning before giving the final answer.

Question: {question}

Options:
{options_text}

Please follow these steps:
1. Analyze the question and understand what is being asked
2. Consider each option carefully with medical knowledge
3. Provide your reasoning step by step
4. End your response with "The correct answer is: [LETTER]" where [LETTER] is STRICTLY one latter in [A,B,C,D].

Your response:"""
    return prompt

def apply_qwen_template(prompt: str, tokenizer) -> str:
    # If using actual tokenizer with chat template
    if tokenizer and hasattr(tokenizer, 'apply_chat_template'):
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True, # Note: This param is specific to some models
        )
        return text
    else:
        # Fallback raw prompt
        return prompt

# Test Prompt Generation
if dataset:
    sample_q = dataset[0]["question"]
    sample_opt = dataset[0]["options"]
    raw_prompt = create_prompt(sample_q, sample_opt)
    
    print("--- Raw User Prompt ---\n")
    print(raw_prompt)
    
    # final_input = apply_qwen_template(raw_prompt, tokenizer)
    # print("\n--- Final Input to Model (Simulated) ---\n")
    # print(final_input)

In [ ]:
@torch.inference_mode()
def generate_answers_transformers(
    model,
    tokenizer,
    prompts: List[str],
    max_new_tokens: int = 512,
    temperature: float = 0.6,
    top_p: float = 0.9,
):
    """
    Simulated batch generation if model is None, else runs actual generation.
    """
    if model is None or tokenizer is None:
        print("Model not loaded, skipping generation.")
        return ["This is a dummy response. <think>Thinking...</think> <answer>The correct answer is: C.</answer>"] * len(prompts)

    inputs_text = [apply_qwen_template(p, tokenizer) for p in prompts]

    batch = tokenizer(
        inputs_text,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    first_device = next(iter(model.parameters())).device
    batch = {k: v.to(first_device) for k, v in batch.items()}

    do_sample = temperature > 0

    print(f"Generating with Temp={temperature}, Do_Sample={do_sample}...")

    gen_ids = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_lens = batch["attention_mask"].sum(dim=1).tolist()
    texts = []
    for i, out_ids in enumerate(gen_ids):
        gen_part = out_ids[prompt_lens[i]:]
        texts.append(tokenizer.decode(gen_part, skip_special_tokens=False))
    return texts

In [ ]:
# 答案提取逻辑调试
_RE_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_RE_ANSWER_BLOCK = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)

_STRONG_PATTERNS = [
    re.compile(r"The\s+correct\s+answer\s+is\s*[:：]?\s*(?:\*\*|\*)?\s*([A-D])\b", re.IGNORECASE),
    re.compile(r"Final\s+answer\s*[:：]?\s*(?:\*\*|\*)?\s*([A-D])\b", re.IGNORECASE),
    re.compile(r"\bAnswer\s*[:：]?\s*(?:\*\*|\*)?\s*([A-D])\b", re.IGNORECASE),
    re.compile(r"The\s+correct\s+answer\s+is\s*[:：]?\s*(?:\*\*|\*)?\s*([A-D])\s*[.)]\s*", re.IGNORECASE),
]

_RE_LASTLINE_LETTER = re.compile(r"^\s*(?:option\s*)?([A-D])\s*$", re.IGNORECASE)
_RE_LASTLINE_LETTER_PUNCT = re.compile(r"^\s*(?:option\s*)?([A-D])\s*[.)]\s*$", re.IGNORECASE)

def extract_final_answer(response: str, tail_chars: int = 1500) -> Optional[str]:
    if not response:
        return None

    # Step 1: Remove <think> content
    text = _RE_THINK.sub("", response)

    # Step 2: Try <answer> block
    m = _RE_ANSWER_BLOCK.search(text)
    scope = m.group(1) if m else text[-tail_chars:]

    print(f"DEBUG: Searching scope: '{scope.strip()}'")

    # Step 3: Strong Patterns
    for pat in _STRONG_PATTERNS:
        matches = list(pat.finditer(scope))
        if matches:
            res = matches[-1].group(1).upper()
            print(f"DEBUG: Match found via Strong Pattern: {res}")
            return res

    # Step 4: Fallback line check
    lines = [ln.strip() for ln in scope.splitlines() if ln.strip()]
    for ln in reversed(lines[-8:]):
        mm = _RE_LASTLINE_LETTER.match(ln) or _RE_LASTLINE_LETTER_PUNCT.match(ln)
        if mm:
            res = mm.group(1).upper()
            print(f"DEBUG: Match found via Last Line Fallback: {res}")
            return res

    print("DEBUG: No match found.")
    return None

# Test inputs
test_responses = [
    "So the conclusion is clear. The correct answer is: B",
    "<think>Maybe A? No.</think> <answer>C</answer>",
    "Analysis... Therefore, option D is correct.", # This might fail with current Strong Patterns if not strictly "The correct answer is"
    "The correct answer is: **A**.",
    "B", # Simple fallback
]

print("\n--- Testing Extractor ---")
for resp in test_responses:
    print(f"\nResponse: {resp}")
    ans = extract_final_answer(resp)
    print(f"Extracted: {ans}")

In [ ]:
def evaluate_model(
    model,
    tokenizer,
    dataset,
    max_samples: Optional[int] = None,
    batch_size: int = 1,
    max_new_tokens: int = 512,
    temperature: float = 0.6,
    top_p: float = 0.9,
):
    correct = 0
    total = 0
    results = []

    if max_samples:
        dataset = dataset[:max_samples]

    items: List[Tuple[int, str, dict, str]] = []
    for idx, item in enumerate(dataset):
        question = item.get("question", "")
        options = item.get("options", {})
        gt = item.get("answer_idx", "")
        if not gt:
             # Try other fields if answer_idx is missing
             gt = item.get("answer", "")
        gt = gt.upper().strip()

        if not question or not options or not gt:
            print(f"Skipping Invalid Item {idx}: missing fields")
            continue
        items.append((idx, question, options, gt))

    for start in tqdm(range(0, len(items), batch_size), desc="Evaluating"):
        chunk = items[start : start + batch_size]
        prompts = [create_prompt(q, opt) for (_, q, opt, _) in chunk]

        responses = generate_answers_transformers(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        for (orig_idx, question, options, gt), resp in zip(chunk, responses):
            pred = extract_final_answer(resp)
            is_correct = (pred == gt)
            if is_correct:
                correct += 1
            total += 1

            results.append(
                {
                    "id": orig_idx,
                    "question": question,
                    "ground_truth": gt,
                    "predicted_answer": pred,
                    "response": resp,
                    "is_correct": is_correct,
                }
            )
            print(f"Item {orig_idx}: GT={gt}, Pred={pred} -> {'CORRECT' if is_correct else 'WRONG'}")

    acc = correct / total if total else 0.0
    return {"accuracy": acc, "correct": correct, "total": total, "results": results}

# 运行评估（Demo）
print("\n--- Running Evaluation Loop ---")
eval_stats = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    max_samples=2, # 只跑 2 个作为测试
    temperature=TEMPERATURE,
    top_p=TOP_P
)
print(f"\nFinal Accuracy: {eval_stats['accuracy']:.2%}")

In [ ]:
def save_results(results, output_path: str, model_name: str = ""):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent="\t", ensure_ascii=False)
    print(f"\nFull results saved to {output_path}")

    summary_path = output_path.replace(".json", "_summary.txt")
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write("MEDICAL QA EVALUATION SUMMARY\n")
        f.write("=" * 50 + "\n")
        if model_name:
            f.write(f"Model: {model_name}\n")
        f.write("Dataset: MedQA\n")
        f.write(f"Total questions: {results['total']}\n")
        f.write(f"Correct answers: {results['correct']}\n")
        f.write(f"Accuracy: {results['accuracy']:.4f} ({results['accuracy']:.2%})\n")
        f.write("=" * 50 + "\n")
    print(f"Summary saved to {summary_path}")

save_results(eval_stats, OUTPUT_FILE, model_name="Qwen3-14B-Demo")